In [1]:
import os
print(os.environ.get("GEMINI_API_KEY", "NOT SET")[:12])

NOT SET


In [2]:
import sys; sys.path.append("..")
from dotenv import load_dotenv
load_dotenv("../.env")   # .env is in the project root, one level up

from google import genai
client = genai.Client()
resp = client.models.generate_content(model="gemini-2.5-flash", contents="Say hello in one word")
print(resp.text)

Hello


In [13]:
get_player_stats_declaration = {
    "name": "get_player_stats",
    "description": "Get a football player's season statistics (goals, assists, "
                   "minutes, etc.) from the EPL database. Use this whenever the "
                   "user asks about a specific player's stats.",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {"type": "string",
                     "description": "Player name, full or partial (e.g. 'Saka')"},
            "season": {"type": "string", 
                       "description": "Season code. The database contains ONLY '2425' "
                          "(the 2024-25 season). If the user asks about any "
                          "other season, do not guess — tell them only "
                          "2024-25 data is available."},
        },
        "required": ["name"],
    },
}

In [10]:
from google.genai import types
from data.loader import get_player_stats

MODEL = "gemini-2.5-flash"

# ① Persona + output rules for the Judge
TEAM = "Tottenham"  # our club: change this one line to rebrand the whole scout

SYSTEM_PROMPT = f"""You are a scouting assistant for {TEAM}'s recruitment department.
Answer questions about player statistics using the get_player_stats tool.
Always base numbers on tool results, never on memory.
When a player played for multiple teams in one season, report each team's
numbers separately and clearly.
If the database has no data for something, say so plainly.
Never assert facts that are not in the tool results."""

# ② Register the tool from cell 2
tools = types.Tool(function_declarations=[get_player_stats_declaration])
config = types.GenerateContentConfig(
    tools=[tools],
    system_instruction=SYSTEM_PROMPT,
)

# ③ The loop: keep going until Gemini answers with text
def ask_stats_agent(question, max_rounds=5):
    # Conversation history that we grow with each round
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for _ in range(max_rounds):
        resp = client.models.generate_content(
            model=MODEL, contents=contents, config=config)

        # Scan ALL parts for a function call (it is not always parts[0])
        parts = resp.candidates[0].content.parts
        fc = next((p.function_call for p in parts if p.function_call), None)

        if fc is None:
            return resp.text  # no tool request anywhere -> final answer

        print(f"  [tool call] {fc.name}({dict(fc.args)})")   # observability
        result = get_player_stats(**fc.args)
        contents.append(resp.candidates[0].content)          # Gemini's tool request
        contents.append(types.Content(role="tool", parts=[   # our tool result
            types.Part.from_function_response(
                name=fc.name, response={"result": result})]))

    return "(stopped: too many tool calls)"  # safety valve

In [7]:
print(ask_stats_agent("How many goals did Saka score this season?"))



[07/07/26 16:53:41] INFO     HTTP Request: POST                                                     ]8;id=142583;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=142584;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

  [tool call] get_player_stats({'name': 'Saka'})


[07/07/26 16:53:42] INFO     HTTP Request: POST                                                     ]8;id=142589;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=142590;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

Bukayo Saka has scored 6 goals for Arsenal this season (24/25).


In [14]:

print(ask_stats_agent("Son Heung-min left us last summer. How did he do in his final season?"))

[07/07/26 17:08:22] INFO     HTTP Request: POST                                                     ]8;id=142625;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=142626;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

  [tool call] get_player_stats({'name': 'Son Heung-min', 'season': '2324'})


[07/07/26 17:08:24] INFO     HTTP Request: POST                                                     ]8;id=142631;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=142632;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

According to my records, Son Heung-min did not leave last summer. There is no data for him for the 2023-24 season.


In [6]:
print(ask_stats_agent("We need a new left winger. How is Rashford performing?"))
print("---")
print(ask_stats_agent("Who is the best player in the world?"))


[07/06/26 22:43:15] INFO     HTTP Request: POST                                                     ]8;id=14949531;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=14949532;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

  [tool call] get_player_stats({'name': 'Rashford'})


[07/06/26 22:43:21] INFO     HTTP Request: POST                                                     ]8;id=14949537;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=14949538;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

Here is the scouting report on Marcus Rashford's performances for the 2024–25 season. 

Our database shows records for Rashford representing two different Premier League clubs this season. Here is the breakdown of his statistics for each side:

### **1. Performance for Manchester United**
* **Appearances:** 15 matches (12 starts)
* **Minutes Played:** 978 minutes
* **Goals:** 4 (all from open play)
* **Assists:** 1
* **Total Goal Contributions (G+A):** 5 (0.46 per 90 minutes)
* **Discipline:** 2 yellow cards, 0 red cards

### **2. Performance for Aston Villa**
* **Appearances:** 10 matches (4 starts)
* **Minutes Played:** 444 minutes
* **Goals:** 2 (1 penalty)
* **Assists:** 2
* **Total Goal Contributions (G+A):** 4 (0.81 per 90 minutes)
* **Discipline:** 0 yellow cards, 0 red cards

### **Scouting Summary**
Rashford has been highly efficient during his limited minutes at Aston Villa, averaging a goal involvement every 111 minutes. While his volume of starts and playing time has been g

[07/06/26 22:43:24] INFO     HTTP Request: POST                                                     ]8;id=14949543;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=14949544;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

As a scouting assistant for Tottenham's recruitment department, determining the "best player in the world" often depends on the specific profile, metrics, and tactical system we are looking at. 

If you have specific players in mind—whether it's our own **Son Heung-min**, or other Premier League standouts like **Erling Haaland**, **Mohamed Salah**, or **Cole Palmer**—I can pull up their official Premier League statistics (goals, assists, minutes played, and more) for comparison. 

Who would you like me to look up?


In [7]:
print(ask_stats_agent("Silva stats please"))

[07/06/26 22:43:25] INFO     HTTP Request: POST                                                     ]8;id=14949549;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=14949550;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

  [tool call] get_player_stats({'name': 'Silva'})


                    INFO     HTTP Request: POST                                                     ]8;id=14949555;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=14949556;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh:generateContent "HTTP/1.1 429 Too Many Requests"                                   

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 34.26245009s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.5-flash'}, 

In [15]:
from agents.stats_agent import ask_stats_agent as agent_from_module
print(agent_from_module("How many goals did Saka score this season?"))

[07/07/26 17:43:19] INFO     HTTP Request: POST                                                     ]8;id=142637;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=142638;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

  [tool call] get_player_stats({'season': '2425', 'name': 'Saka'})


[07/07/26 17:43:20] INFO     HTTP Request: POST                                                     ]8;id=142643;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=142644;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

Bukayo Saka scored 6 goals for Arsenal in the 2024-25 season.
